# Lab 8: Ekspertni sistemi – Backward Chaining i Baza Pitanja

Ovaj notebook pokriva:
- **Pravila lančanja unazad** (`use` / `when` – PyKE ekvivalent)
- **Bazu pitanja** (interaktivni unos umjesto hardkodiranih činjenica)
- Sve 4 zadatka iz Lab 8


---
## 0. Backward Chaining Engine (PyKE ekvivalent)

Implementacija slijedi PyKE model:
- `BCRule` – pravilo s `use` (zaključak) i `when` (uvjeti)
- `QuestionBase` – baza pitanja, pita korisnika i kešira odgovore
- `BCEngine` – motor koji dokazuje ciljeve pretragom unazad

Algoritam dokazivanja cilja `prove(P, *args)`:
1. Pretraži bazu činjenica
2. Ako postoji pitanje za predikat `P` → upita korisnika
3. Za svako pravilo čiji `use` odgovara cilju → pokušaj dokazati sve `when` uvjete


In [ ]:
# ─── Backward Chaining Engine ───────────────────────────────────────────────

class BCRule:
    """Pravilo lančanja unazad: use zaključak  when *uvjeti"""
    def __init__(self, name, conc_pred, conc_args, conditions):
        self.name       = name
        self.conc_pred  = conc_pred    # string – naziv predikata zaključka
        self.conc_args  = conc_args    # tuple – None = wildcard, '$var' = varijabla
        self.conditions = conditions   # lista (pred, args_tuple)


class QuestionBase:
    """Baza pitanja (.kqb ekvivalent) – pita korisnika i kešira odgovore."""

    def __init__(self):
        self._q     = {}   # predicate -> dict
        self._cache = {}   # cached answers

    def add_yn(self, pred, text):
        """Dodaje Yes/No pitanje."""
        self._q[pred] = {'type': 'yn', 'text': text}

    def add_select(self, pred, text, choices):
        """Dodaje pitanje s višestrukim izborom."""
        self._q[pred] = {'type': 'select', 'text': text, 'choices': choices}

    def has(self, pred):
        return pred in self._q

    def ask(self, pred, prefill=None):
        """Pita korisnika (ili koristi prefill za automatsko testiranje)."""
        if pred in self._cache:
            return self._cache[pred]
        q = self._q[pred]
        if prefill and pred in prefill:
            raw = prefill[pred]
            result = raw
            label = f"y" if raw is True else ("n" if raw is False else str(raw))
            print(f"  [AUTO] {q['text']} → {label}")
        else:
            print(f"\n  {q['text']}")
            if q['type'] == 'yn':
                while True:
                    a = input("    (y/n): ").strip().lower()
                    if a in ('y', 'n'):
                        result = (a == 'y')
                        break
                    print("    Unesite 'y' ili 'n'.")
            else:
                for i, c in enumerate(q['choices'], 1):
                    print(f"    {i}. {c}")
                while True:
                    try:
                        result = int(input("    Vaš izbor: ").strip())
                        if 1 <= result <= len(q['choices']):
                            break
                    except ValueError:
                        pass
                    print(f"    Unesite 1–{len(q['choices'])}.")
        self._cache[pred] = result
        return result

    def reset(self):
        self._cache.clear()


class BCEngine:
    """Backward chaining engine – PyKE bc_example ekvivalent."""

    def __init__(self):
        self.facts = set()
        self.rules = []
        self.qb    = QuestionBase()
        self._prefill = None   # za automatsko testiranje

    def reset(self):
        self.facts = set()
        self.qb.reset()

    def assert_fact(self, pred, *args):
        self.facts.add((pred,) + tuple(args))

    def add_rule(self, rule):
        self.rules.append(rule)

    # ── public ────────────────────────────────────────────────────────────────

    def prove_goal(self, pred, *pattern, prefill=None):
        """Vraća listu binding dicts za sve načine dokazivanja cilja."""
        self._prefill = prefill
        return list(self._prove(pred, pattern, {}))

    # ── internals ─────────────────────────────────────────────────────────────

    def _apply(self, pattern, bindings):
        out = []
        for p in pattern:
            if isinstance(p, str) and p.startswith('$') and p != '$_':
                out.append(bindings.get(p[1:], p))
            else:
                out.append(p)
        return tuple(out)

    def _unify(self, pattern, args, bindings):
        if len(pattern) != len(args):
            return None
        nb = dict(bindings)
        for p, a in zip(pattern, args):
            if p is None or p == '$_':
                continue
            if isinstance(p, str) and p.startswith('$'):
                var = p[1:]
                if var in nb:
                    if nb[var] != a:
                        return None
                else:
                    nb[var] = a
            elif p != a:
                return None
        return nb

    def _prove(self, pred, pattern, bindings):
        applied = self._apply(pattern, bindings)

        # 1. Baza činjenica
        for fact in self.facts:
            if fact[0] == pred:
                b = self._unify(applied, fact[1:], bindings)
                if b is not None:
                    yield b

        # 2. Baza pitanja
        if self.qb.has(pred):
            answer = self.qb.ask(pred, self._prefill)
            b = self._unify(applied, (answer,), bindings)
            if b is not None:
                yield b

        # 3. Backward chaining pravila
        for rule in self.rules:
            if rule.conc_pred == pred:
                b = self._unify(applied, rule.conc_args, bindings)
                if b is not None:
                    yield from self._prove_conds(rule.conditions, b)

    def _prove_conds(self, conditions, bindings):
        if not conditions:
            yield bindings
            return
        (pred, args), *rest = conditions
        applied = self._apply(args, bindings)
        for sub in self._prove(pred, applied, bindings):
            merged = {**bindings, **sub}
            yield from self._prove_conds(rest, merged)


print("BCEngine spreman.")

---
# Zadatak 1 – Detaljno dokazivanje: Bruce i Jim su prvi rođaci

## Teorijsko objašnjenje

Cilj koji treba dokazati: **`cousins(bruce, jim, 1)`**

Pravilo koje se koristi:
```
first_cousins
  use cousins($cousin1, $cousin2, 1)
  when
    child_parent($cousin1, $sibling1, $_, $_)    ← uvjet 1
    siblings($sibling1, $sibling2, $_, $_)        ← uvjet 2
    child_parent($cousin2, $sibling2, $_, $_)    ← uvjet 3
```
Bindinzi: `cousin1=bruce`, `cousin2=jim`.

### Uvjet 1: `child_parent(bruce, $sibling1, $_, $_)`

PyKE pronalazi 4 pravila tipa `child_parent`. Za `bruce` (koji je sin) relevantan je:
```
father_son
  use child_parent($child, $father, father, son)
  when family.son_of($child, $father, $mother)
mother_son
  use child_parent($child, $mother, mother, son)
  when family.son_of($child, $father, $mother)
```
Provjera u bazi činjenica: `son_of(bruce, thomas, norma)` → postoji!
- Pravilo `father_son` → `sibling1 = thomas`
- Pravilo `mother_son` → `sibling1 = norma`

**Zaključak uvjeta 1:** `sibling1 ∈ {thomas, norma}`

### Uvjet 2: `siblings($sibling1, $sibling2, $_, $_)`

Pravilo za siblinge provjerava zajednički roditeljski odnos.

**Za `sibling1 = thomas`** (dijete peter-a i ann-e):
- U bazi činjenica: `son_of(thomas, peter, ann)` i `daughter_of(joyce, peter, ann)`, `daughter_of(phyllis, peter, ann)`
- `sibling2 ∈ {lucy, joyce, phyllis}` (ostali koji dijele iste roditelje)

**Za `sibling1 = norma`** (dijete abner-a i anna-e):
- U bazi činjenica: `daughter_of(norma, abner, anna)`, `son_of(homer, abner, anna)`, `son_of(john_w, abner, anna)`, `son_of(bill, abner, anna)`, `son_of(chuck_w, abner, anna)`, `daughter_of(shirley, abner, anna)`
- `sibling2 ∈ {homer, john_w, bill, chuck_w, shirley}`

**Zaključak uvjeta 2:** za svaki par `(sibling1, sibling2)` postoji kombinacija.

### Uvjet 3: `child_parent(jim, $sibling2, $_, $_)`

Potrebno dokazati da je `jim` dijete nekog od pronađenih `sibling2` vrijednosti.

Provjera u bazi činjenica: `son_of(jim, bill, pat)` → **postoji!**
- Dakle, jim je sin od `bill`-a.
- Ovo odgovara slučaju gdje `sibling1 = norma` i `sibling2 = bill`.

**Zaključak uvjeta 3:** `sibling2 = bill` → **uvjet dokazan!**

### Finalni zaključak
Sve tri činjenice dokazane:
1. `child_parent(bruce, norma, mother, son)` ✓
2. `siblings(norma, bill, ...)` ✓ (norma i bill su djeca abner-a i anna-e)
3. `child_parent(jim, bill, father, son)` ✓

→ **`cousins(bruce, jim, 1)` je dokazano!** Bruce i Jim jesu prvi rođaci.


In [ ]:
# ── Proširena baza činjenica (uključuje proširenu porodicu za dokaz prvog rođaštva)

EXTENDED_FACTS = [
    # Originalna porodica (Lab 7)
    ('son_of',      'bruce',   'thomas',  'norma'),
    ('son_of',      'fred',    'thomas',  'norma'),
    ('daughter_of', 'anne',    'thomas',  'norma'),
    ('son_of',      'homer',   'abner',   'anna'),
    ('daughter_of', 'norma',   'abner',   'anna'),
    ('daughter_of', 'shirley', 'abner',   'anna'),
    ('son_of',      'thomas',  'peter',   'ann'),
    ('daughter_of', 'lucy',    'peter',   'ann'),
    ('son_of',      'brian',   'walter',  'elise'),
    ('daughter_of', 'holly',   'brian',   'shirley'),
    ('son_of',      'mark',    'brian',   'shirley'),
    ('daughter_of', 'jan',     'brian',   'shirley'),
    # Proširena porodica za cousins primjer
    ('daughter_of', 'joyce',   'peter',   'ann'),    # thomas-ova sestra
    ('daughter_of', 'phyllis', 'peter',   'ann'),    # thomas-ova sestra
    ('son_of',      'john_w',  'abner',   'anna'),   # norma-in brat
    ('son_of',      'bill',    'abner',   'anna'),   # norma-in brat (otac jim-a)
    ('son_of',      'chuck_w', 'abner',   'anna'),   # norma-in brat
    ('son_of',      'jim',     'bill',    'pat'),    # jim → otac = bill
]

# ── Forward chaining engine (iz Lab 7) s proširenim pravilima ────────────────

class KnowledgeEngine:
    def __init__(self):
        self.facts = set()

    def reset(self):
        self.facts = set()

    def assert_fact(self, predicate, *args):
        self.facts.add((predicate,) + tuple(args))

    def query(self, predicate, *pattern):
        results = []
        for fact in self.facts:
            if fact[0] != predicate:
                continue
            args = fact[1:]
            if len(args) != len(pattern):
                continue
            if all(p is None or p == a for p, a in zip(pattern, args)):
                results.append(fact)
        return results

    def activate(self, rules):
        changed = True
        while changed:
            changed = False
            for rule in rules:
                for fact in rule(self):
                    if fact not in self.facts:
                        self.facts.add(fact)
                        changed = True


def fc_son_of(eng):
    new = []
    for f in eng.query('son_of', None, None, None):
        _, child, father, mother = f
        new += [('child_parent', child, father, 'father', 'son'),
                ('child_parent', child, mother, 'mother', 'son')]
    return new

def fc_daughter_of(eng):
    new = []
    for f in eng.query('daughter_of', None, None, None):
        _, child, father, mother = f
        new += [('child_parent', child, father, 'father', 'daughter'),
                ('child_parent', child, mother, 'mother', 'daughter')]
    return new

def fc_sibling(eng):
    new = []
    for f1 in eng.query('child_parent', None, None, None, None):
        _, c1, parent, _, _ = f1
        for f2 in eng.query('child_parent', None, parent, None, None):
            _, c2, _, _, _ = f2
            if c1 != c2:
                new.append(('sibling', c1, c2))
    return new

def fc_cousins(eng):
    """Dokazuje prvi stupanj rođaštva – isti algoritam kao BC first_cousins pravilo."""
    new = []
    for s_fact in eng.query('sibling', None, None):
        _, s1, s2 = s_fact
        for cp1 in eng.query('child_parent', None, s1, None, None):
            _, c1, _, _, _ = cp1
            for cp2 in eng.query('child_parent', None, s2, None, None):
                _, c2, _, _, _ = cp2
                new.append(('cousins', c1, c2, 1))
    return new

EXT_RULES = [fc_son_of, fc_daughter_of, fc_sibling, fc_cousins]

fc_eng = KnowledgeEngine()
for fact in EXTENDED_FACTS:
    fc_eng.assert_fact(*fact)
fc_eng.activate(EXT_RULES)
print(f"Ukupno činjenica: {len(fc_eng.facts)}")

In [ ]:
# ── Korak-po-korak trace dokaza ───────────────────────────────────────────────

SEP = '─' * 60

def cousins_proof_trace(person1='bruce', person2='jim'):
    print(f"CILJ: cousins({person1}, {person2}, 1)")
    print("Pravilo: first_cousins")
    print(f"  use cousins($c1, $c2, 1)")
    print(f"  when")
    print(f"    child_parent($c1, $s1, $_, $_)  ← uvjet 1")
    print(f"    siblings($s1, $s2, $_, $_)       ← uvjet 2")
    print(f"    child_parent($c2, $s2, $_, $_)  ← uvjet 3")
    print(f"Bind: c1={person1}, c2={person2}")

    # ── UVJET 1 ────────────────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print(f"UVJET 1: child_parent({person1}, $s1, $_, $_)")
    print(SEP)
    cp1_results = fc_eng.query('child_parent', person1, None, None, None)
    if not cp1_results:
        print(f"  ✗ Nije pronađen nijedan child_parent za {person1}.")
        return
    s1_values = set()
    for _, _, parent, ptype, ctype in cp1_results:
        print(f"  ✓ child_parent({person1}, {parent}, {ptype}, {ctype}) – pronađeno pravilom")
        if ctype == 'son':
            print(f"      primjena pravila father_son / mother_son na son_of({person1}, ...)")
        else:
            print(f"      primjena pravila father_daughter / mother_daughter na daughter_of({person1}, ...)")
        s1_values.add(parent)
    print(f"  → s1 ∈ {sorted(s1_values)}")

    # ── UVJET 2 ────────────────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print(f"UVJET 2: siblings($s1, $s2, $_, $_)")
    print(SEP)
    s2_by_s1 = {}
    for s1 in sorted(s1_values):
        sibs = [f[2] for f in fc_eng.query('sibling', s1, None)]
        s2_by_s1[s1] = sibs
        print(f"  Za s1={s1}: sibling({s1}, ?) → s2 ∈ {sorted(sibs)}")
        for s2 in sibs:
            print(f"      ✓ sibling({s1}, {s2}) – zajednički roditelj")

    # ── UVJET 3 ────────────────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print(f"UVJET 3: child_parent({person2}, $s2, $_, $_)")
    print(SEP)
    cp2_results = fc_eng.query('child_parent', person2, None, None, None)
    p2_parents  = {r[2] for r in cp2_results}
    print(f"  Roditelji osobe {person2}: {sorted(p2_parents)}")

    found = False
    for s1, s2_list in s2_by_s1.items():
        for s2 in s2_list:
            if s2 in p2_parents:
                print(f"\n  *** MATCH: s1={s1}, s2={s2} ***")
                print(f"  ✓ child_parent({person2}, {s2}, ...) pronađeno u bazi činjenica!")
                print(f"  Dokaz: {s1} (roditelj od {person1}) i {s2} (roditelj od {person2}) su brat/sestra.")
                found = True

    print(f"\n{SEP}")
    if found:
        print(f"ZAKLJUČAK: cousins({person1}, {person2}, 1) je DOKAZANO! ✓")
        print(f"  {person1} i {person2} su prvi rođaci.")
    else:
        print(f"ZAKLJUČAK: cousins({person1}, {person2}, 1) NIJE dokazano. ✗")

cousins_proof_trace('bruce', 'jim')

In [ ]:
# ── Provjera: je li cousins(bruce, jim, 1) u bazi izvedenih činjenica? ─────
result = fc_eng.query('cousins', 'bruce', 'jim', 1)
print("Provjera direktnim upitom u izvedenu bazu činjenica:")
if result:
    print(f"  ✓ {result[0]} – Bruce i Jim JESU prvi rođaci.")
else:
    print("  ✗ Nije pronađeno.")

---
# Zadatak 2 – Vremenski ES sa backward chaining pravilima

Isti sistem iz Lab 7 (Zadatak 2), ali sada sa pravilima **lančanja unazad** (`use`/`when`).

Pravila:
```
kabanica_rule
  use ponijeti('kabanica')
  when
    pada_kisa(True)
    puse_vjetar(True)

kissobran_rule
  use ponijeti('kissobran')
  when
    pada_kisa(True)
    puse_vjetar(False)

nista_rule
  use ponijeti('nista')
  when
    pada_kisa(False)
    puse_vjetar(False)
```
Razlika od forward chaining: sistem **ne aktivira** sva pravila unaprijed, nego dokazuje cilj `ponijeti($item)` na zahtjev.


In [ ]:
# ── Baza pravila (backward chaining) za vremenski ES ─────────────────────────

WEATHER_BC_RULES = [
    BCRule('kabanica_rule',
           'ponijeti', ('kabanica',),
           [('pada_kisa',   (True,)),
            ('puse_vjetar', (True,))]),

    BCRule('kissobran_rule',
           'ponijeti', ('kissobran',),
           [('pada_kisa',   (True,)),
            ('puse_vjetar', (False,))]),

    BCRule('nista_rule',
           'ponijeti', ('nista',),
           [('pada_kisa',   (False,)),
            ('puse_vjetar', (False,))]),
]


def weather_bc_driver(pada_kisa: bool, puse_vjetar: bool):
    """Driver za vremenski ES sa backward chaining pravilima."""
    eng = BCEngine()
    for rule in WEATHER_BC_RULES:
        eng.add_rule(rule)

    # Baza činjenica (.kfb)
    eng.assert_fact('pada_kisa',   pada_kisa)
    eng.assert_fact('puse_vjetar', puse_vjetar)

    # Postavljanje cilja: Šta ponijeti?
    results = eng.prove_goal('ponijeti', '$item')
    if results:
        for b in results:
            print(f"  Trebate ponijeti: {b['item'].capitalize()}")
    else:
        print("  Nema preporuke za ovaj slučaj.")


print("--- Scenarij 1: kiša=True,  vjetar=True  ---")
weather_bc_driver(pada_kisa=True,  puse_vjetar=True)

print("\n--- Scenarij 2: kiša=True,  vjetar=False ---")
weather_bc_driver(pada_kisa=True,  puse_vjetar=False)

print("\n--- Scenarij 3: kiša=False, vjetar=False ---")
weather_bc_driver(pada_kisa=False, puse_vjetar=False)

print("\n--- Scenarij 4: kiša=False, vjetar=True  ---")
weather_bc_driver(pada_kisa=False, puse_vjetar=True)

---
# Zadatak 3 – Vremenski ES sa bazom pitanja

Baza činjenica se **ne koristi**. Umjesto nje, korisnik odgovara na dva Yes/No pitanja:
- *Da li pada kiša?*
- *Da li je vjetrovito?*

Baza pravila ostaje ista kao u Zadatku 2, samo se sada uvjeti dokazuju kroz bazu pitanja.

PyKE `.kqb` ekvivalent:
```
pada_kisa($ans)
  Da li pada kiša?
  --
  $ans = yn

puse_vjetar($ans)
  Da li je vjetrovito?
  --
  $ans = yn
```


In [ ]:
def weather_questions_driver(prefill=None):
    """Driver za vremenski ES sa bazom pitanja (bez baze činjenica).
    prefill: dict za automatsko testiranje, npr. {'pada_kisa': True, 'puse_vjetar': False}
    """
    eng = BCEngine()

    # Baza pravila – iste BC regule
    for rule in WEATHER_BC_RULES:
        eng.add_rule(rule)

    # Baza pitanja (.kqb) – nema baze činjenica!
    eng.qb.add_yn('pada_kisa',   'Da li pada kiša?')
    eng.qb.add_yn('puse_vjetar', 'Da li je vjetrovito?')

    # Postavljanje cilja
    results = eng.prove_goal('ponijeti', '$item', prefill=prefill)
    if results:
        for b in results:
            print(f"  Trebate ponijeti: {b['item'].capitalize()}")
    else:
        print("  Nema preporuke za ovaj slučaj.")


# ── Automatski testovi (prefill zamjenjuje input() za potrebe verifikacije) ──
print("=== Test 1: kiša=True,  vjetar=True  ===")
weather_questions_driver({'pada_kisa': True, 'puse_vjetar': True})

print("\n=== Test 2: kiša=True,  vjetar=False ===")
weather_questions_driver({'pada_kisa': True, 'puse_vjetar': False})

print("\n=== Test 3: kiša=False, vjetar=False ===")
weather_questions_driver({'pada_kisa': False, 'puse_vjetar': False})

print("\n=== Test 4: kiša=False, vjetar=True  ===")
weather_questions_driver({'pada_kisa': False, 'puse_vjetar': True})

In [ ]:
# ── Interaktivni mod (ukloni '#' ispod i pokreni za pravi unos) ──────────────
# weather_questions_driver()

---
# Zadatak 4 – Vremenski ES u doba korone sa pitanjima

Dodaju se dva nova eksperta i jedno **multiple choice** pitanje:

- **Ekspert 1**: ako je poplava → ponesi *gumene čizme*
- **Ekspert 2**: ako je pandemija → ponesi *masku*

`.kqb` ekvivalent novog pitanja:
```
situacija($ans)
  Šta se trenutno vani dešava?
  --
  $ans = select_1
  1: Poplava
  2: Pandemija
  3: Ništa posebno
```

Sistem ispisuje **dvije poruke**:
1. Šta ponijeti zbog vremenskih uvjeta (kiša/vjetar)
2. Šta ponijeti zbog posebne situacije (poplava/pandemija)


In [ ]:
# ── Dodatna BC pravila za posebne situacije ───────────────────────────────────

CORONA_BC_RULES = [
    BCRule('poplava_rule',
           'posebno_ponijeti', ('gumene_cizme',),
           [('situacija', (1,))]),       # 1 = poplava

    BCRule('pandemija_rule',
           'posebno_ponijeti', ('maska',),
           [('situacija', (2,))]),       # 2 = pandemija

    BCRule('normalno_rule',
           'posebno_ponijeti', ('nista_posebno',),
           [('situacija', (3,))]),       # 3 = ništa posebno
]


def weather_corona_driver(prefill=None):
    """Driver za vremenski ES u doba korone.
    prefill: dict, npr. {'pada_kisa': True, 'puse_vjetar': False, 'situacija': 2}
    """
    eng = BCEngine()

    # Baza pravila – vremenski + corona
    for rule in WEATHER_BC_RULES + CORONA_BC_RULES:
        eng.add_rule(rule)

    # Baza pitanja (.kqb) – bez baze činjenica
    eng.qb.add_yn('pada_kisa',   'Da li pada kiša?')
    eng.qb.add_yn('puse_vjetar', 'Da li je vjetrovito?')
    eng.qb.add_select('situacija',
                      'Šta se trenutno vani dešava?',
                      ['Poplava', 'Pandemija', 'Ništa posebno'])

    # ── Poruka 1: kiša/vjetar
    print("  -- Poruka 1 (vremenski uvjeti) --")
    results1 = eng.prove_goal('ponijeti', '$item', prefill=prefill)
    if results1:
        for b in results1:
            print(f"  Trebate ponijeti: {b['item'].capitalize()}")
    else:
        print("  Nema preporuke za vremenski uvjete.")

    # ── Poruka 2: posebna situacija
    print("  -- Poruka 2 (posebna situacija) --")
    results2 = eng.prove_goal('posebno_ponijeti', '$item', prefill=prefill)
    if results2:
        for b in results2:
            item = b['item'].replace('_', ' ').capitalize()
            print(f"  Zbog posebne situacije, ponesite i: {item}")
    else:
        print("  Nema posebnih preporuka.")


print("=== Test A: kiša=True, vjetar=True, situacija=Poplava ===")
weather_corona_driver({'pada_kisa': True, 'puse_vjetar': True, 'situacija': 1})

print("\n=== Test B: kiša=True, vjetar=False, situacija=Pandemija ===")
weather_corona_driver({'pada_kisa': True, 'puse_vjetar': False, 'situacija': 2})

print("\n=== Test C: kiša=False, vjetar=False, situacija=Ništa posebno ===")
weather_corona_driver({'pada_kisa': False, 'puse_vjetar': False, 'situacija': 3})

print("\n=== Test D: kiša=False, vjetar=True, situacija=Pandemija ===")
weather_corona_driver({'pada_kisa': False, 'puse_vjetar': True, 'situacija': 2})

In [ ]:
# ── Interaktivni mod (ukloni '#' ispod i pokreni za pravi unos) ──────────────
# weather_corona_driver()

---
## Sažetak razlika: Forward vs Backward Chaining

| Osobina | Forward Chaining (Lab 7) | Backward Chaining (Lab 8) |
|---|---|---|
| Ključne riječi | `foreach` / `assert` | `use` / `when` |
| Smjer pretrage | Unaprijed (od činjenica ka cilju) | Unazad (od cilja ka činjenicama) |
| Aktivacija pravila | Da – sva pravila se izvršavaju | Ne – samo relevantna pravila |
| Baza pitanja | Nije korištena | Podržana (`.kqb`) |
| Efikasnost | Može generisati nepotrebne činjenice | Fokusiran samo na cilj |
